In [0]:
# DDP MNIST-ish toy training on synthetic data (no I/O)
import os
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader, DistributedSampler

# Tiny synthetic dataset
class ToyDataset(Dataset):
    def __init__(self, n=50_000, in_dim=784, n_classes=10):
        g = torch.Generator().manual_seed(0)
        self.x = torch.randn(n, in_dim, generator=g)
        self.y = torch.randint(0, n_classes, (n,), generator=g)
    def __len__(self): return self.x.size(0)
    def __getitem__(self, i): return self.x[i], self.y[i]

# Tiny MLP
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=512, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, n_classes)
        )
    def forward(self, x): return self.net(x)

def train_main(epochs=2, batch_size=128, lr=1e-3):
    # TorchDistributor exports torchrun-style env vars
    rank        = int(os.environ["RANK"])
    world_size  = int(os.environ["WORLD_SIZE"])
    local_rank  = int(os.environ["LOCAL_RANK"])

    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl", init_method="env://", world_size=world_size, rank=rank)

    # (Optional) make NCCL a bit more verbose when debugging
    # os.environ.setdefault("NCCL_ASYNC_ERROR_HANDLING", "1")
    # os.environ.setdefault("NCCL_DEBUG", "WARN")

    # Data
    dataset = ToyDataset()
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, drop_last=True)
    loader  = DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)

    # Model / Optim
    model = MLP().to(local_rank)
    model = DDP(model, device_ids=[local_rank])
    opt   = optim.AdamW(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()

    # Train
    for epoch in range(epochs):
        sampler.set_epoch(epoch)
        model.train()
        running = 0.0
        for i, (x, y) in enumerate(loader):
            x = x.to(local_rank, non_blocking=True)
            y = y.to(local_rank, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = lossf(logits, y)
            loss.backward()
            opt.step()
            running += loss.item()

            if (i + 1) % 50 == 0 and rank == 0:
                print(f"[epoch {epoch+1}] step {i+1}: loss={running/50:.4f}")
                running = 0.0

    # Clean up
    dist.destroy_process_group()
    if rank == 0:
        print("✅ Training complete.")

In [0]:
from pyspark.ml.torch.distributor import TorchDistributor

# 4 nodes × 4 GPUs per node = 16 processes
result = TorchDistributor(
    num_processes=16,     # total processes == total GPUs
    local_mode=False,     # multi-node
    use_gpu=True
).run(train_main, epochs=2, batch_size=128, lr=1e-3)